# SLayer in 10 minutes

An agent that writes raw SQL can produce a perfectly valid query for the *wrong* metric — and lose the context behind every choice the next time it's invoked. SLayer takes a different bet: give the agent a typed map of the data, metrics that compose at query time, and a memory of the business context it has accumulated. The result is queries that stay consistent across calls and an agent that gets sharper every session, not blanker.

**Audience:** anyone building agents that touch data. **Time:** ~10 minutes.

## What an agent needs to actually analyse data

1. **See** what data is available
2. **Get** trusted metric definitions
3. **Ask** for ad-hoc queries — including complex ones
4. **Know** which metric to choose when
5. **Have** business context for specific cases
6. **Find** the relevant bits of all of the above quickly

The rest of the talk walks through each one.

## Setup

We'll use the bundled **Jaffle Shop** dataset — synthetic coffee-shop orders across 6 stores, 3 years of data, ~960k orders. Every cell below runs against it.

**Prerequisites:** `pip install 'motley-slayer[embedding_search]' jafgen`. `OPENAI_API_KEY` is *optional* — without it, search still works via BM25 + tantivy; with it, the dense-embedding channel also activates.

In [1]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', '..'))
sys.path.insert(0, os.getcwd())

from setup_talk import ensure_lightning_talk_demo
from slayer.async_utils import run_sync
from slayer.core.errors import MemoryNotFoundError

engine, storage, client, models = ensure_lightning_talk_demo()
print(f'{len(models)} models ingested:')
for m in sorted(models, key=lambda m: m.name):
    print(f'  - {m.name:>10}  ({len(m.columns)} columns, {len(m.joins)} joins)')

7 models ingested:
  -  customers  (2 columns, 0 joins)
  -      items  (3 columns, 2 joins)
  -     orders  (7 columns, 2 joins)
  -   products  (5 columns, 0 joins)
  -     stores  (4 columns, 0 joins)
  -   supplies  (5 columns, 1 joins)
  -     tweets  (4 columns, 1 joins)


## See — auto-ingestion gave us models for free

Point SLayer at a database, it walks the schema, infers types, and discovers foreign keys. The agent gets a typed graph of your tables without anyone writing YAML. The setup cell above already did this — there's nothing else to configure.

See [Auto-Ingestion docs](../../concepts/ingestion.md).

In [2]:
orders = next(m for m in models if m.name == 'orders')
print(f'orders: {len(orders.columns)} columns, {len(orders.joins)} direct joins')
print()
print('Columns:')
for col in orders.columns:
    pk = ' [PK]' if col.primary_key else ''
    print(f'  {col.name:<14} {str(col.type):<10}{pk}')
print()
print('Joins discovered automatically:')
for j in orders.joins:
    src, tgt = j.join_pairs[0]
    print(f'  orders.{src} -> {j.target_model}.{tgt}')

orders: 7 columns, 2 direct joins

Columns:
  id             TEXT       [PK]
  customer_id    TEXT      
  ordered_at     DATE      
  store_id       TEXT      
  subtotal       DOUBLE    
  tax_paid       DOUBLE    
  order_total    DOUBLE    

Joins discovered automatically:
  orders.customer_id -> customers.id
  orders.store_id -> stores.id


## Get — one metric definition, the right rollup per question

`order_total` is the metric source — defined once in the model. The **aggregation** (sum / average / median / weighted-average / …) is the agent's choice **at query time**, picked per question without redefining what 'revenue' means. The metric never drifts across calls because the definition lives in the model; the agent isn't forced to pre-commit to one rollup.

In [3]:
result = engine.execute_sync(query={
    'source_model': 'orders',
    'measures': [
        {'formula': 'order_total:sum',    'name': 'revenue'},
        {'formula': 'order_total:avg',    'name': 'avg_ticket'},
        {'formula': 'order_total:median', 'name': 'median_ticket'},
    ],
    'dimensions': ['stores.name'],
    'order': [{'column': 'revenue', 'direction': 'desc'}],
})
assert len(result.data) > 0, 'multi-aggregation query returned no rows'
for row in result.data:
    print(
        f"  {row['orders.stores.name']:<16} "
        f"revenue=${row['orders.revenue']:>11,.2f}  "
        f"avg=${row['orders.avg_ticket']:>6,.2f}  "
        f"median=${row['orders.median_ticket']:>6,.2f}"
    )

  Brooklyn         revenue=$2,813,316.42  avg=$ 10.94  median=$  6.24
  Philadelphia     revenue=$2,090,196.59  avg=$ 10.85  median=$  6.36
  Chicago          revenue=$1,172,850.83  avg=$ 11.02  median=$  6.37
  San Francisco    revenue=$1,043,370.30  avg=$ 11.17  median=$  6.44
  New Orleans      revenue=$ 160,643.11  avg=$ 10.19  median=$  6.24


## Ask — complex queries compose like Lego

Real questions stack things: group by month, aggregate, compare to the prior month, compare to last year, filter to growth, take the top N. Doing this in raw SQL means CTEs, window functions, self-joins. SLayer lets the agent build the question out of primitives.

In [4]:
hero = {
    'source_model': 'orders',
    'measures': [
        {'formula': 'order_total:sum', 'name': 'revenue'},
        {'formula': 'change_pct(order_total:sum)', 'name': 'mom_growth'},
        {
            'formula': "order_total:sum / time_shift(order_total:sum, -1, 'year') - 1",
            'name': 'yoy_growth',
        },
    ],
    'dimensions': ['stores.name'],
    'time_dimensions': [{'dimension': 'ordered_at', 'granularity': 'month'}],
    'filters': ['change_pct(order_total:sum) > 0'],
    'order': [{'column': 'mom_growth', 'direction': 'desc'}],
    'limit': 10,
}
result = engine.execute_sync(query=hero)
assert len(result.data) > 0, 'hero query returned no rows'
print(f"{'Month':<10} {'Store':<16} {'Revenue':>11} {'MoM':>9} {'YoY':>9}")
print('-' * 58)
for row in result.data:
    month = str(row['orders.ordered_at'])[:7]
    yoy = row.get('orders.yoy_growth')
    yoy_s = f'{yoy:>8.1%}' if yoy is not None else '       -'
    print(
        f"{month:<10} {row['orders.stores.name']:<16} "
        f"${row['orders.revenue']:>10,.0f} "
        f"{row['orders.mom_growth']:>8.1%} {yoy_s}"
    )

Month      Store                Revenue       MoM       YoY
----------------------------------------------------------
2023-06    Philadelphia     $    19,310   766.7%        -
2025-02    Chicago          $    30,635   164.3%        -
2023-08    Philadelphia     $    33,613    41.3%        -
2025-07    San Francisco    $    70,832    39.9%        -
2025-05    San Francisco    $    39,730    34.0%        -
2024-07    Brooklyn         $    95,180    29.7%        -
2025-07    Chicago          $    76,732    29.0%        -
2025-06    San Francisco    $    50,637    27.5%        -
2025-06    Chicago          $    59,490    27.4%        -
2024-07    Philadelphia     $    69,061    23.4%   190.4%


In [5]:
print(result.sql)

SELECT
    "orders.stores.name",
    "orders.ordered_at",
    "orders.revenue",
    "orders.mom_growth",
    "orders.yoy_growth"
FROM (
SELECT *
FROM (
WITH base AS (
SELECT
  stores.name AS "orders.stores.name",
  DATE_TRUNC('MONTH', orders.ordered_at) AS "orders.ordered_at",
  SUM(orders.order_total) AS "orders.revenue"
FROM orders AS orders
LEFT JOIN stores AS stores
  ON orders.store_id = stores.id
GROUP BY
  stores.name,
  DATE_TRUNC('MONTH', orders.ordered_at)
),
shifted__ts_mom_growth AS (
SELECT
  stores.name AS "orders.stores.name",
  DATE_TRUNC('MONTH', CAST(orders.ordered_at + INTERVAL '1' MONTH AS TIMESTAMP)) AS "orders.ordered_at",
  SUM(orders.order_total) AS "orders.revenue"
FROM orders AS orders
LEFT JOIN stores AS stores
  ON orders.store_id = stores.id
GROUP BY
  stores.name,
  DATE_TRUNC('MONTH', CAST(orders.ordered_at + INTERVAL '1' MONTH AS TIMESTAMP))
),
sjoin__ts_mom_growth AS (
SELECT base."orders.ordered_at", base."orders.revenue", base."orders.stores.name", sh

## Queries as models — unlimited depth

Every SLayer query also defines a model: its result columns become dimensions, its numeric columns become measure sources, and the next query can `source_model` it by name. No bolt-on multi-stage syntax — this falls out of the design.

Example: *average monthly revenue per store*. Stage 1 sums revenue by store-month; stage 2 averages those monthly sums per store.

In [6]:
result = engine.execute_sync(query=[
    {
        'name': 'monthly_store_revenue',
        'source_model': 'orders',
        'measures': ['order_total:sum'],
        'dimensions': ['stores.name'],
        'time_dimensions': [{'dimension': 'ordered_at', 'granularity': 'month'}],
    },
    {
        'source_model': 'monthly_store_revenue',
        'measures': ['order_total_sum:avg'],
        'dimensions': ['stores__name'],
        'order': [{'column': 'order_total_sum_avg', 'direction': 'desc'}],
    },
])
assert len(result.data) > 0, 'multistage query returned no rows'
print(f"{'Store':<16} {'Avg monthly revenue':>22}")
print('-' * 40)
for row in result.data:
    print(
        f"  {row['monthly_store_revenue.stores__name']:<14} "
        f"${row['monthly_store_revenue.order_total_sum_avg']:>20,.2f}"
    )

Store               Avg monthly revenue
----------------------------------------
  Brooklyn       $           93,777.21
  Chicago        $           68,991.23
  San Francisco  $           65,210.64
  Philadelphia   $           56,491.80
  New Orleans    $           26,773.85


## Have & Find — text context plus a way to find it

Most semantic layers stop at a `description:` field per metric and leave the agent to guess which one to read. SLayer ships two things on top:

- **Memories** — free-form notes the agent writes, each linked to whichever entities it's about (or carrying an example query).
- **Search** — one tool that retrieves both memories and canonical entities, fusing three retrieval channels.

See [Memories docs](../../concepts/memories.md) and [Search docs](../../concepts/search.md).

In [7]:
brooklyn = run_sync(client.save_memory(
    learning=(
        'Brooklyn switched to a new POS system in late 2024. '
        'Order totals before 2025-01-01 are from the legacy system; '
        'they exclude tax and use a different currency-rounding rule. '
        'Exclude pre-2025-01-01 Brooklyn data when comparing to other stores.'
    ),
    linked_entities=[
        'jaffle_shop.orders.order_total',
        'jaffle_shop.stores.name',
    ],
    id='lightning.brooklyn_pos',
))
print(f'Saved memory id={brooklyn.memory_id}')
print(f'Linked to:       {brooklyn.resolved_entities}')
if brooklyn.warnings:
    print(f'Warnings:        {brooklyn.warnings}')

Saved memory id=lightning.brooklyn_pos
Linked to:       ['jaffle_shop.orders.order_total', 'jaffle_shop.stores.name']


In [8]:
top_customers = run_sync(client.save_memory(
    learning=(
        'Top-5 customers by lifetime spend - known-good query pattern '
        'used in the weekly CRM-ops dashboard.'
    ),
    linked_entities={
        'source_model': 'orders',
        'measures': [{'formula': 'order_total:sum', 'name': 'lifetime_spend'}],
        'dimensions': ['customers.name'],
        'order': [{'column': 'lifetime_spend', 'direction': 'desc'}],
        'limit': 5,
    },
    id='lightning.top_customers',
))
print(f'Saved query-bearing memory id={top_customers.memory_id}')
print(f'Auto-extracted entities:      {top_customers.resolved_entities}')

Saved query-bearing memory id=lightning.top_customers
Auto-extracted entities:      ['jaffle_shop.orders', 'jaffle_shop.customers.name', 'jaffle_shop.orders.order_total']


## Find — one tool, three channels

`search` retrieves both memories and canonical entities, merging three channels via Reciprocal Rank Fusion:

1. **BM25** over each memory's stored entity tags — strongest when the agent already has an entity reference in hand.
2. **tantivy** full-text over learning text *and* canonical entities — natural-language questions.
3. **embeddings** (cosine over a sidecar table) — dense similarity. Optional; degrades gracefully if `OPENAI_API_KEY` isn't set.

The agent doesn't pick a channel. It asks once.

In [9]:
resp = run_sync(client.search(
    question='What should I know before comparing Brooklyn revenue to other stores?',
    max_memories=3,
    max_example_queries=2,
    max_entities=0,
))
print('Memories:')
for hit in resp.memories:
    print(f'  [{hit.id}]  score={hit.score:.3f}')
    print(f'     -> {hit.text[:100]}...')
assert any(m.id == 'lightning.brooklyn_pos' for m in resp.memories), \
    'Brooklyn memory must surface in resp.memories for this question'

Memories:
  [lightning.brooklyn_pos]  score=0.016
     -> Brooklyn switched to a new POS system in late 2024. Order totals before 2025-01-01 are from the lega...


In [10]:
resp = run_sync(client.search(
    entities=['jaffle_shop.orders.order_total'],
    max_memories=3,
    max_entities=0,
))
print('Memories anchored to jaffle_shop.orders.order_total:')
for hit in resp.memories:
    print(f'  [{hit.id}]  matched_entities={hit.matched_entities}')
    print(f'     -> {hit.text[:100]}...')
assert any(m.id == 'lightning.brooklyn_pos' for m in resp.memories), \
    'Brooklyn memory must surface via the order_total entity tag'

Memories anchored to jaffle_shop.orders.order_total:
  [lightning.brooklyn_pos]  matched_entities=['jaffle_shop.orders.order_total']
     -> Brooklyn switched to a new POS system in late 2024. Order totals before 2025-01-01 are from the lega...


In [11]:
resp = run_sync(client.search(
    question='How have analysts queried customer lifetime spend before?',
    max_memories=0,
    max_example_queries=2,
    max_entities=5,
))
print('Example queries:')
for eq in resp.example_queries:
    print(f'  [{eq.id}]  score={eq.score:.3f}')
    print(f'     -> {eq.text[:100]}...')
print()
print('Canonical entities:')
for ent in resp.entities:
    print(f'  [{ent.kind}]  {ent.id}  (score={ent.score:.3f})')
assert any(eq.id == 'lightning.top_customers' for eq in resp.example_queries), \
    'top-customers example query must surface in resp.example_queries'
assert len(resp.entities) > 0, 'expected at least one canonical entity hit'

Example queries:
  [lightning.top_customers]  score=0.016
     -> Top-5 customers by lifetime spend - known-good query pattern used in the weekly CRM-ops dashboard.
T...

Canonical entities:
  [column]  jaffle_shop.orders.customer_id  (score=0.016)
  [model]  jaffle_shop.customers  (score=0.016)
  [model]  jaffle_shop.orders  (score=0.016)
  [column]  jaffle_shop.customers.name  (score=0.016)
  [column]  jaffle_shop.customers.id  (score=0.015)


## Wrap-up

**Here today:** open source (MIT). Interfaces: MCP, CLI, REST, Python, Flight SQL. Auto-ingestion. Postgres / MySQL / Snowflake / BigQuery / DuckDB / ClickHouse / SQLite. Multistage queries, named measures, custom aggregations, memories with embedding search.

**Coming next:** proper graph / Cypher support, so memories and entities form an explicit graph the agent can traverse.

### Try it in Claude Code

One command wires SLayer up as an MCP server with the Jaffle Shop demo preloaded:

```bash
claude mcp add slayer -- uvx --from motley-slayer slayer mcp --demo
```

Then ask Claude in any project: *"What stores are in jaffle_shop and which one has the highest revenue?"* — it will call the same tools used above.

Links: [README](https://github.com/MotleyAI/slayer) · [Discord](https://discord.gg/egWxMctHCA) · [Docs](https://motley-slayer.readthedocs.io/).

In [12]:
# Idempotent teardown: both ids may already be absent (e.g. partial prior
# run, or this cell rerun standalone). MemoryNotFoundError is the only
# expected failure mode here.
try:
    run_sync(client.forget_memory('lightning.brooklyn_pos'))
except MemoryNotFoundError:
    pass
try:
    run_sync(client.forget_memory('lightning.top_customers'))
except MemoryNotFoundError:
    pass
print('Memories removed. Re-running this notebook is idempotent.')

Memories removed. Re-running this notebook is idempotent.
